# E04a — smaller main model (validation only)

**Runtime:** Runtime → Change runtime type → **L4 GPU**. Then Runtime → Run all; about 10–20 minutes. Keep the tab open.

**One change vs E03, fixed before any run (experiment ledger, E04a):** model size only.
Width 64→32, Transformer layers 2→1, feed-forward 128→64, per-input summary 64→16 → **472,636 → 44,908 weights**.
Everything else as E03: 32 channels (18 price + 11 E03 inputs + 3 news), sector label, patches 5 and 20, RevIN,
lr 1e-4, dropout 0.2, max 30 epochs, patience 10, seeds 20261015/16/17.

A check cell builds both models and stops the notebook unless the weight counts and output shapes are exactly as expected.
Outputs in `MyDrive/Senior/results/`: `main_model_e04a.json` + `_preds.csv.gz`, `results_summary_e04a.xlsx`.


In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT_DIR = f'{SENIOR}/prices', f'{SENIOR}/results'

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib
src = open('scripts/baselines_deep.py', encoding='utf-8').read()
print('baselines_deep.py', hashlib.sha256(open('scripts/baselines_deep.py', 'rb').read()).hexdigest()[:12])
assert 'multiscale_sector_small' in src, 'stop: push the latest baselines_deep.py (with multiscale_sector_small) from your PC'


In [ ]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo "$OUT_DIR"
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
# Files must match the PC copies (hashes recorded 2026-10-15; sentiment file as stored in git, LF line endings)
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
# Pipeline and look-ahead tests; the notebook stops here if either fails
import subprocess
for t in ['tests/test_pipeline.py', 'tests/test_no_lookahead.py']:
    r = subprocess.run(['python', t], capture_output=True, text=True)
    print('\n'.join(l for l in r.stdout.splitlines() if 'defined on' not in l)[-1500:])
    assert r.returncode == 0 and 'FAIL' not in r.stdout, f'stop: {t} failed'


In [ ]:
# Model check: weight counts and output shapes, before any training (stops on mismatch)
import sys, torch
sys.path.insert(0, 'scripts'); sys.path.insert(0, 'src')
import baselines_deep as BD
expected = {'multiscale_sector': 472_636, 'multiscale_sector_small': 44_908}   # 32 channels, 18 sectors
for name, n in expected.items():
    m = BD.make_model(name, 32, 18, 0.2)
    got = sum(p.numel() for p in m.parameters())
    out = m(torch.randn(4, 60, 32), torch.randint(0, 19, (4,)))
    print(f'{name}: {got:,} weights; trend {list(out["trend_logits"].shape)}, returns {list(out["returns"].shape)}')
    assert got == n, f'stop: {name} has {got:,} weights, expected {n:,}'
    assert list(out['trend_logits'].shape) == [4, 3, 3] and list(out['returns'].shape) == [4, 3], 'stop: wrong output shape'
print('model check passed')


In [ ]:
# E04a: small model, E03 inputs and settings
!python scripts/baselines_deep.py --models multiscale_sector_small --news data/derived/sentiment_scores.csv --extra-features e03 \
    --lr 1e-4 --dropout 0.2 --max-epochs 30 --patience 10 \
    --out "$OUT_DIR/main_model_e04a.json"


In [ ]:
# Quick look: best epochs, inner hold-out, validation macro-F1 vs E03 (9-seed single) and the pass bar
import json, statistics as st
r = json.load(open(f'{OUT_DIR}/main_model_e04a.json'))
print(r['git_commit'], r['script_sha256'][:12], len(r['settings']['channels']), 'channels')
runs = r['models']['multiscale_sector_small']['runs']
for run in runs:
    print('seed', run['seed'], 'weights', run['params'], 'best epoch', run['best_epoch'], 'of', len(run['history']),
          'inner hold-out', run['best_inner_es_macro_f1'], {h: round(run['val'][h]['macro_f1'], 3) for h in ('h1', 'h5', 'h10')})
e03 = {'h1': 0.338, 'h5': 0.352, 'h10': 0.348}
bar = {'h1': 0.393, 'h5': 0.400, 'h10': 0.407}
for h in bar:
    m = st.mean(run['val'][h]['macro_f1'] for run in runs)
    print(h, 'E04a', round(m, 4), '| E03 (9 seeds)', e03[h], '| diff', round(m - e03[h], 4), '| bar', bar[h])


In [ ]:
# Side-by-side spreadsheet: E03 vs E04a
!python scripts/summarize_results.py --inputs "$OUT_DIR/main_model_e03.json" "$OUT_DIR/main_model_e04a.json" --out "$OUT_DIR/results_summary_e04a.xlsx"
